# Yield Prediction Model

This notebook trains a separate crop yield prediction model from `../../yield_data/yield_df.csv` and saves it to `../../models/yield_model.pkl`. It does not use or modify the disease model.

## 1. Imports

In [1]:
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Load Yield Dataset

In [2]:
DATA_PATH = Path("../../yield_data/yield_df.csv")
MODEL_PATH = Path("../../models/yield_model.pkl")
METRICS_PATH = Path("../../models/model_metrics.json")

df = pd.read_csv(DATA_PATH).drop(columns=["Unnamed: 0"], errors="ignore").dropna()
print(df.shape)
df.head()

(28242, 7)


,Area,Item,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,Albania,Maize,1990,36613,1485.0,121.0,16.37
1,Albania,Potatoes,1990,66667,1485.0,121.0,16.37
2,Albania,"Rice, paddy",1990,23333,1485.0,121.0,16.37
3,Albania,Sorghum,1990,12500,1485.0,121.0,16.37
4,Albania,Soybeans,1990,7000,1485.0,121.0,16.37


## 3. Train Model

In [3]:
features = ["Area", "Item", "Year", "average_rain_fall_mm_per_year", "pesticides_tonnes", "avg_temp"]
target = "hg/ha_yield"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), ["Area", "Item"]),
    ("numeric", StandardScaler(), ["Year", "average_rain_fall_mm_per_year", "pesticides_tonnes", "avg_temp"]),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
])

model.fit(X_train, y_train)
print("Yield model training complete.")

Yield model training complete.


## 4. Evaluate Model

In [4]:
predictions = model.predict(X_test)
metrics = {
    "rows": int(len(df)),
    "features": features,
    "target": target,
    "test_size": int(len(X_test)),
    "mae_hg_per_ha": float(mean_absolute_error(y_test, predictions)),
    "rmse_hg_per_ha": float(mean_squared_error(y_test, predictions) ** 0.5),
    "r2": float(r2_score(y_test, predictions)),
}
metrics

{'rows': 28242,
 'features': ['Area',
  'Item',
  'Year',
  'average_rain_fall_mm_per_year',
  'pesticides_tonnes',
  'avg_temp'],
 'target': 'hg/ha_yield',
 'test_size': 5649,
 'mae_hg_per_ha': 3452.627510178793,
 'rmse_hg_per_ha': 9419.223489095491,
 'r2': 0.9877687092146042}

## 5. Save Model

In [5]:
MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump(model, MODEL_PATH, compress=3)

all_metrics = {}
if METRICS_PATH.exists():
    all_metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
all_metrics["yield_model"] = metrics
METRICS_PATH.write_text(json.dumps(all_metrics, indent=2), encoding="utf-8")

print(f"Saved yield model to: {MODEL_PATH}")

Saved yield model to: ..\..\models\yield_model.pkl


## 6. Example Prediction

In [6]:
sample = X_test.iloc[[0]].copy()
predicted_yield = model.predict(sample)[0]
print("Input:")
display(sample)
print(f"Predicted yield: {predicted_yield:.2f} hg/ha")
print(f"Actual yield: {y_test.iloc[0]:.2f} hg/ha")

Input:


,Area,Item,Year,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
25564,Spain,"Rice, paddy",2008,636.0,40719.0,17.21


Predicted yield: 71358.92 hg/ha
Actual yield: 69220.00 hg/ha
